# Surprisal scoring: Llama-3.1-8B, Llama-3.1-70B, Qwen-2.5-72B

Word-level surprisal in bits at the reflexive via shifted cross-entropy (`-ln P / ln 2`).

Llama-3.1-8B is loaded in float16. Llama-3.1-70B is the official checkpoint in 4-bit NF4 via bitsandbytes. Qwen-2.5-72B is the Unsloth 4-bit NF4 checkpoint. Both large models were run on a 40GB GPU.

Gated Llama checkpoints need a Hugging Face login. In Colab, store the token as a secret named `HF_TOKEN` — do not paste it into this notebook.

In [ ]:
%pip install -q transformers accelerate bitsandbytes pandas torch huggingface_hub

In [ ]:
import math
import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if token:
    login(token=token)

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA = ROOT / "data"
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

STIMULI = [
    DATA / "exp1_locality.csv",
    DATA / "exp2_hierarchy.csv",
    DATA / "exp3_logophoric.csv",
]


def get_target_surprisal(model, tokenizer, sentence, target_word):
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = inputs.input_ids[:, 1:].contiguous()
    surprisals = (
        F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="none",
        )
        / math.log(2)
    ).view(shift_labels.size())[0].tolist()
    tokens = [tokenizer.decode([token_id]) for token_id in shift_labels[0]]
    for token_str, score in reversed(list(zip(tokens, surprisals))):
        if token_str.strip().replace(".", "").lower() == target_word.lower():
            return score
    if tokens[-1].strip() == ".":
        return surprisals[-2]
    return surprisals[-1]


def score_model(model_id, outfile_tag, **load_kwargs):
    print(f"Loading {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    model.eval()
    names = [
        f"results_exp1_{outfile_tag}.csv",
        f"results_exp2_{outfile_tag}.csv",
        f"results_exp3_{outfile_tag}.csv",
    ]
    for stim_path, out_name in zip(STIMULI, names):
        df = pd.read_csv(stim_path)
        df["surprisal"] = [
            get_target_surprisal(model, tokenizer, row.sentence, row.target)
            for row in df.itertuples(index=False)
        ]
        out_path = OUT / out_name
        df.to_csv(out_path, index=False)
        print("wrote", out_path)
    del model
    torch.cuda.empty_cache()

In [ ]:
score_model(
    "meta-llama/Llama-3.1-8B",
    "llama8b",
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

score_model(
    "meta-llama/Llama-3.1-70B",
    "llama70b",
    quantization_config=bnb_config,
    device_map={"": 0},
)

In [ ]:
score_model(
    "unsloth/Qwen2.5-72B-bnb-4bit",
    "qwen72b",
    device_map={ "": 0 },
    dtype=torch.bfloat16,
)